# Citi Velocity curves and swaption cubes: EOD and intraday timeseries

Multi-currency swap-curve timeseries at **end-of-day** and at **1-minute intraday**,
plus a **swaption-cube** timeseries, all served from warmed local stores.

## Nothing here touches Excel

Every cell below reads a **CurveStore / SwaptionCubeStore partition on disk**. That
matters for more than speed: it works when Excel is closed, it is reproducible (the
same request returns the same curve), and it cannot disturb a signed-in add-in.

The live Excel path still exists and is what warms these stores - see
`citivelo_excel.ipynb` for that, and `scripts/citivelo_excel_intraday_warm.py` for
the warm itself.

## What is warmed

| asset | contents |
|---|---|
| `<curve>-CITIVELOEXCEL` | one EOD curve per day |
| `<curve>-CITIVELOEXCELMIN` | one curve per published **minute** |
| `USD-SWAPTIONVOL-CITIVELOEXCEL` | the swaption cube, ATM + strike offsets |

Curves cover **USD / EUR / GBP / CAD / JPY** over roughly two years; the swaption
cube goes back to **2019**.

In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
plt.style.use("ggplot")
pylab.rcParams.update({
    "legend.fontsize": "medium", "figure.figsize": (18, 6),
    "axes.labelsize": "medium", "axes.titlesize": "medium",
    "xtick.labelsize": "medium", "ytick.labelsize": "medium",
})

import datetime
import warnings

import numpy as np
import pandas as pd
import pytz

NYC_tz = pytz.timezone("America/New_York")
warnings.filterwarnings("ignore", category=UserWarning)

import sys
sys.path.append("../../")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from RVUtils.plt_timeseries import make_secondary_axis_plot

In [2]:
from Caching.curve_store import CurveStore
from Caching.swaption_cube_store import SwaptionCubeStore
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder

CURVES = ["USD-SOFR-1D", "EUR-ESTR-1D", "GBP-SONIA-1D", "CAD-CORRA-1D", "JPY-TONAR-1D-LCH"]

# One MDP, one router, reused by every cell. IRSwapsTB caches the MDP's fetcher
# state, and the fixings a curve needs are resolved once per process rather than
# once per curve - which used to be 97% of a warmed read.
curve_mdp = IRSwapsMDP(source="citivelo_excel_rl")      # ..._ql for a QuantLib curve
irs_tb = IRSwapsTB(curve_mdp, show_tqdm=False, use_ts_cache=False)
ts = TimeseriesBuilder()

## 1. What is actually warmed

Check coverage before asking for a window - a request outside it silently falls
through to the live Excel path, which is slow and needs a signed-in add-in.

In [3]:
store = CurveStore.default()

rows = []
for curve in CURVES:
    for label, suffix in (("EOD", "CITIVELOEXCEL"), ("1-min", "CITIVELOEXCELMIN")):
        asset = f"{curve}-{suffix}"
        try:
            days = sorted(store.available_dates(asset))
        except Exception:
            days = []
        rows.append({
            "curve": curve, "grid": label, "days": len(days),
            "first": days[0] if days else None, "last": days[-1] if days else None,
        })
coverage = pd.DataFrame(rows)
coverage

,curve,grid,days,first,last
0,USD-SOFR-1D,EOD,5506,2005-01-03,2026-08-07
1,USD-SOFR-1D,1-min,998,2022-08-31,2026-08-07
2,EUR-ESTR-1D,EOD,5565,2005-01-04,2026-08-07
3,EUR-ESTR-1D,1-min,527,2024-08-01,2026-08-07
4,GBP-SONIA-1D,EOD,4067,2010-11-26,2026-08-07
5,GBP-SONIA-1D,1-min,527,2024-08-01,2026-08-07
6,CAD-CORRA-1D,EOD,2820,2012-01-03,2026-08-07
7,CAD-CORRA-1D,1-min,632,2024-08-01,2026-08-07
8,JPY-TONAR-1D-LCH,EOD,1108,2017-11-09,2026-08-07
9,JPY-TONAR-1D-LCH,1-min,527,2024-08-01,2026-08-07


## 2. EOD, multiple currencies

The ordinary `TimeseriesBuilder` path. A **bare `datetime.date`** is what selects
the end-of-day curve.

> `pandas.Timestamp` subclasses `datetime` subclasses `date`, so
> `pd.Timestamp("2026-07-01")` is *midnight* and also reads as EOD. Anything with a
> real time-of-day routes to the minute store instead.

In [5]:
queries = [
    UnifiedQuery(curve=c, tenor="10Y", value=UnifiedValue.IRS_RATE, name=f"{c} 10Y")
    for c in ["USD-SOFR-1D", "EUR-ESTR-1D", "GBP-SONIA-1D"]
]

eod = ts.get_timeseries(
    start=datetime.date(2025, 1, 1),
    end=datetime.date(2026, 7, 31),
    queries=queries,
    routers={"IRS": irs_tb},
    ignore_cache_miss=True,
)
eod

,EUR-ESTR-1D 10Y,GBP-SONIA-1D 10Y,USD-SOFR-1D 10Y
Date,,,
2025-01-01,2.21415,4.07115,NaN
2025-01-02,2.22281,4.08025,4.07384
2025-01-03,2.27528,4.09088,4.09298
2025-01-06,2.31393,4.11082,4.13481
2025-01-07,2.34613,4.16296,4.20250
...,...,...,...
2026-07-27,2.92582,4.56676,4.21882
2026-07-28,2.90609,4.53995,4.18693
2026-07-29,2.94204,4.60058,4.20542


In [6]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
for col in eod.columns:
    plot(eod[col], which="left")
legend(valfmt="{:.3f}", show_date=True)

### Curve trades read the same way

`tenor="2y/10y"` is a curve, `"2y/5y/10y"` a fly. The weights and signs are applied
by the query, not by you.

In [7]:
spreads = ts.get_timeseries(
    start=datetime.date(2026, 6, 1),
    end=datetime.date(2026, 7, 31),
    queries=[
        UnifiedQuery(curve=c, tenor="2y/10y", value=UnifiedValue.IRS_RATE, name=f"{c} 2s10s")
        for c in ["USD-SOFR-1D", "EUR-ESTR-1D", "GBP-SONIA-1D"]
    ],
    routers={"IRS": irs_tb},
    ignore_cache_miss=True,
)
# IRS_RATE is a DECIMAL for a 1-leg query and already in BP for 2-3 legs, so
# this needs no scaling. Multiplying by 100 here reported EUR 2s10s as
# 2,539 bp instead of 25.4 - the exact trap section 5 warns about.
spreads.tail()

,EUR-ESTR-1D 2s10s,GBP-SONIA-1D 2s10s,USD-SOFR-1D 2s10s
Date,,,
2026-07-27,25.398000,28.322,5.076
2026-07-28,26.866000,30.926,6.367
2026-07-29,24.547999,27.010,12.216
2026-07-30,28.975000,34.364,15.874
2026-07-31,28.217000,34.308,17.718


## 3. Intraday, multiple currencies

Pass a **timezone-aware** datetime and the request routes to the minute store.

Two things worth knowing:

* the add-in stamps **every** curve in New York wall-clock whatever the currency,
  but partitions are keyed by the curve's **local** trading date - the conversion
  is done for you, so just pass an aware timestamp in any zone;
* `meta_data["snapshot_lag_seconds"]` reports how far the served minute sits from
  the one you asked for. Check it rather than assuming an exact hit.

In [17]:
start = NYC_tz.localize(datetime.datetime(2026, 7, 29, 4, 0))
end   = NYC_tz.localize(datetime.datetime(2026, 7, 29, 17, 0))

queries = [ 
		# 	UnifiedQuery(
        #     curve="USD-SOFR-1D",
        #     tenor="IMM_M27xIMM_U27/IMM_M28xIMM_U28",
        #     value=UnifiedValue.IRS_RATE,
        # ),
	UnifiedQuery(
		curve="USD-SOFR-1D",
		tenor="10y",
		value=UnifiedValue.IRS_CITIVELO_SWAP_SPREAD,
		# value=UnifiedValue.IRS_RATE,
	)
]

intraday = ts.get_timeseries(
    start=start, end=end,
    queries=queries,
    freq="1min",
    routers={"IRS": irs_tb},
    ignore_cache_miss=True,
)
print(intraday.shape)
intraday.head()

(781, 1)


,USD-SOFR-1D 10y OUTRIGHT CITIVELO_SWAP_SPREAD
Date,
2026-07-29 04:00:00-04:00,-41.6331
2026-07-29 04:01:00-04:00,-41.6608
2026-07-29 04:02:00-04:00,-41.6287
2026-07-29 04:03:00-04:00,-41.4737
2026-07-29 04:04:00-04:00,-41.4640


In [18]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(intraday["USD-SOFR-1D 10y OUTRIGHT CITIVELO_SWAP_SPREAD"], which="left")
# plot(intraday["EUR-ESTR-1D 10Y"], which="right")
# plot(intraday["GBP-SONIA-1D 10Y"], which="right")
legend(valfmt="{:.4f}", show_date=True)

### One curve, one day, every published minute

The grid is ~12 hours of 1-minute bars per curve per day, in the curve's own local
session.

In [15]:
day = datetime.date(2026, 7, 22)
minutes = ts.get_timeseries(
    start=NYC_tz.localize(datetime.datetime.combine(day, datetime.time(3, 0))),
    end=NYC_tz.localize(datetime.datetime.combine(day, datetime.time(17, 0))),
    queries=[UnifiedQuery(curve="USD-SOFR-1D", tenor="5Y",
                          value=UnifiedValue.IRS_RATE)],
    freq="1min",
    routers={"IRS": irs_tb},
    ignore_cache_miss=True,
)
print(f"{len(minutes)} minute observations")
display(minutes.describe())
display(minutes)

plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(minutes["USD-SOFR-1D 5Y OUTRIGHT RATE"], which="left")
legend(valfmt="{:.4f}", show_date=True)

841 minute observations


,USD-SOFR-1D 5Y OUTRIGHT RATE
count,841.000000
mean,4.102422
std,0.015940
min,4.074820
25%,4.086540
50%,4.103210
75%,4.117150
max,4.126280


,USD-SOFR-1D 5Y OUTRIGHT RATE
Date,
2026-07-22 03:00:00-04:00,4.089650
2026-07-22 03:01:00-04:00,4.089379
2026-07-22 03:02:00-04:00,4.089489
2026-07-22 03:03:00-04:00,4.089241
2026-07-22 03:04:00-04:00,4.089320
...,...
2026-07-22 16:56:00-04:00,4.117310
2026-07-22 16:57:00-04:00,4.117948
2026-07-22 16:58:00-04:00,4.117661


## 4. The swaption cube

`SwaptionCubeStore` holds the raw vol grid per day - **ATM and every strike
offset** - so a cube timeseries is a store read, not a repricing.

The frame is long: one row per `(expiry, tenor, offset_bp)`, with `vol_bp` in
**normal (bp) vol**.

In [10]:
cube_store = SwaptionCubeStore.default()
VOL_ASSET = "USD-SWAPTIONVOL-CITIVELOEXCEL"

cube_dates = sorted(cube_store.available_dates(VOL_ASSET))
print(f"{len(cube_dates)} days  {cube_dates[0]} .. {cube_dates[-1]}")

one_day = cube_store.read_day(VOL_ASSET, cube_dates[-1])
print(f"{one_day.shape[0]} rows for {cube_dates[-1]}")
print("offsets:", sorted(one_day["offset_bp"].unique()))
one_day.head()

1746 days  2019-08-07 .. 2026-08-06
1989 rows for 2026-08-06
offsets: [np.float64(-200.0), np.float64(-100.0), np.float64(-75.0), np.float64(-50.0), np.float64(-25.0), np.float64(-10.0), np.float64(0.0), np.float64(10.0), np.float64(25.0), np.float64(50.0), np.float64(75.0), np.float64(100.0), np.float64(200.0)]


,expiry,tenor,offset_bp,vol_bp,as_of,currency,measure,skew_measure,served_unit,vol_unit,strike_unit,source,citi_index,schema_version,asset,date
0,1M,1Y,-200.0,170.6860,2026-08-06,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-06
1,1M,1Y,-100.0,117.7810,2026-08-06,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-06
2,1M,1Y,-75.0,104.0540,2026-08-06,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-06
3,1M,1Y,-50.0,90.7722,2026-08-06,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-06
4,1M,1Y,-25.0,79.8300,2026-08-06,USD,NORMAL,NORMALABSOLUTE,bp,bp,bp,citivelo_excel_warm/DAILY,USD_SOFR,1,USD-SWAPTIONVOL-CITIVELOEXCEL,2026-08-06


In [11]:
def atm_vol_timeseries(asset, pairs, start, end):
    '''ATM normal vol (bp) per (expiry, tail), one row per day.

    Reads each day once and pulls every requested cell out of it - the store read
    is ~45 ms/day, so the loop is dominated by whatever you do with the frame.
    '''
    days = [d for d in cube_store.available_dates(asset) if start <= d <= end]
    out = []
    for d in days:
        frame = cube_store.read_day(asset, d)
        atm = frame[frame["offset_bp"] == 0]
        row = {"Date": d}
        for expiry, tail in pairs:
            cell = atm[(atm["expiry"] == expiry) & (atm["tenor"] == tail)]
            row[f"{expiry}x{tail} ATM nvol"] = float(cell["vol_bp"].iloc[0]) if len(cell) else np.nan
        out.append(row)
    return pd.DataFrame(out).set_index("Date")


PAIRS = [("3M", "10Y"), ("1Y", "10Y"), ("5Y", "10Y"), ("3M", "2Y")]
atm = atm_vol_timeseries(VOL_ASSET, PAIRS, datetime.date(2024, 1, 1), datetime.date(2026, 8, 6))
print(atm.shape)
atm.tail()

(649, 4)


,3Mx10Y ATM nvol,1Yx10Y ATM nvol,5Yx10Y ATM nvol,3Mx2Y ATM nvol
Date,,,,
2026-07-31,80.0037,83.9343,86.1437,98.4374
2026-08-03,76.9877,82.8158,85.6716,94.8586
2026-08-04,75.1800,81.5841,84.7934,92.0633
2026-08-05,74.2386,80.9518,84.5367,90.9156
2026-08-06,77.2107,82.1221,84.8101,96.3277


In [12]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
for col in atm.columns:
    plot(atm[col], which="left")
legend(valfmt="{:.1f}", show_date=True)

### Skew through time

`offset_bp` is the strike offset from the forward in **basis points**, so the smile
is a slice at fixed `(expiry, tenor)`.

In [13]:
def skew_timeseries(asset, expiry, tail, offsets, start, end):
    days = [d for d in cube_store.available_dates(asset) if start <= d <= end]
    out = []
    for d in days:
        frame = cube_store.read_day(asset, d)
        sl = frame[(frame["expiry"] == expiry) & (frame["tenor"] == tail)]
        row = {"Date": d}
        for off in offsets:
            cell = sl[sl["offset_bp"] == off]
            row[f"{off:+d}bp"] = float(cell["vol_bp"].iloc[0]) if len(cell) else np.nan
        out.append(row)
    frame = pd.DataFrame(out).set_index("Date")
    # Quote the wings against ATM: the level moves far more than the shape.
    for col in frame.columns:
        if col != "+0bp":
            frame[f"{col} - ATM"] = frame[col] - frame["+0bp"]
    return frame


skew = skew_timeseries(VOL_ASSET, "3M", "10Y", [-100, -50, 0, 50, 100],
                       datetime.date(2025, 1, 1), datetime.date(2026, 8, 6))
skew[[c for c in skew.columns if c.endswith("- ATM")]].tail()

,-100bp - ATM,-50bp - ATM,+50bp - ATM,+100bp - ATM
Date,,,,
2026-07-31,17.5464,3.4413,12.4885,30.8563
2026-08-03,18.2988,3.7962,12.7720,31.3763
2026-08-04,16.0303,3.0728,11.5172,28.4970
2026-08-05,16.1926,3.1537,11.5740,28.5884
2026-08-06,15.6564,2.8822,11.4193,28.3343


In [14]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
for col in [c for c in skew.columns if c.endswith("- ATM")]:
    plot(skew[col], which="left")
plot(skew["+0bp"], which="right")
legend(valfmt="{:.2f}", show_date=True)

### The smile on one date

In [15]:
snap = cube_store.read_day(VOL_ASSET, cube_dates[-1])
smile = (snap[(snap["expiry"] == "3M") & (snap["tenor"] == "10Y")]
         .sort_values("offset_bp")[["offset_bp", "vol_bp"]]
         .set_index("offset_bp"))

surface = (snap[snap["offset_bp"] == 0]
           .pivot_table(index="expiry", columns="tenor", values="vol_bp"))

print(f"3Mx10Y smile on {cube_dates[-1]}")
display(smile.T)
print("\nATM surface (normal bp vol)")
display(surface)

3Mx10Y smile on 2026-08-06


offset_bp,-200.0,-100.0,-75.0,-50.0,-25.0,-10.0,0.0,10.0,25.0,50.0,75.0,100.0,200.0
vol_bp,123.501,92.8671,85.883,80.0929,76.73,76.5041,77.2107,78.5697,81.64,88.63,96.843,105.545,140.667



ATM surface (normal bp vol)


tenor,10Y,15Y,1Y,20Y,2Y,30Y,3Y,5Y,7Y
expiry,,,,,,,,,
10Y,83.3969,80.9414,87.2466,80.0696,86.9382,78.7013,86.5244,85.6851,84.7912
12Y,82.1714,79.7898,86.4631,78.7350,86.2477,77.6494,85.6506,84.4465,83.5507
15Y,80.3250,78.0311,85.5241,76.6898,85.3751,76.0192,84.4792,82.6478,81.7134
18M,82.6972,80.2879,99.7998,78.1542,96.2749,76.7593,93.9301,88.8515,85.8710
1M,72.0879,69.0951,76.5882,66.7715,88.1142,64.9960,87.4211,84.5106,79.3128
1Y,82.1221,79.3856,99.0069,76.8825,97.8904,75.3207,95.4030,89.3187,85.9035
20Y,77.6803,75.4938,83.5037,74.3334,83.2697,73.7096,82.1722,79.9377,79.0358
2M,74.8905,71.6160,85.0147,69.2333,92.8936,67.2299,91.7588,86.7007,81.6815
2Y,83.6497,81.4983,98.2533,79.6306,94.6997,78.3278,92.0334,88.5814,86.0731


## 5. Gotchas

Each of these produced a confident wrong number during development.

**Units.** `curve.fair_rate()` returns a **decimal**; Citi quotes **percent**.
A draft of this very notebook scaled the 2s10s cell by 100 and printed EUR 2s10s as
**2,539 bp** instead of 25.4.
Comparing them unscaled reads as a 442 bp error on a curve that is exact. Through
`UnifiedValue.IRS_RATE` a 1-leg query is decimal and a 2-3 leg query is **bp** -
which is why the curve cell above multiplies by 100 and the outright does not.

**Spot lag.** If you build instruments yourself, take the lag from
`SettlementDays`, not `payment_lag`. They coincide for USD (both 2) and diverge for
EUR (1 vs 2); the wrong one reported 0.4358 bp at EUR 3M where the right one gives
0.0006 bp.

**`bulk_get_data` is not a fast path here.** It has branches for CME / ERIS / SDR /
BARCHART but none for CITIVELO, so it falls through to a generic per-point loop -
and for EOD it is *slower* than looping `get_data`. `TimeseriesBuilder` as used
above is the right entry point.

**`ignore_cache=True` reaches Excel.** It bypasses the store and rebuilds through
the cached-then-live quotes layer. Leave it off unless you mean it.

**Coverage is not uniform.** Check section 1 before choosing a window; a date
outside the warm silently takes the live path.